# Step 1: Data Preparation


Here, we explain how to use our code to extract the sub-tomograms which we use to construct the model inputs and targets for model fitting in Step 2. This step is identical in logic to the original tutorial; only the input tomograms and a few config values differ for the BrainWeb data.

In [ ]:
import os
import shutil
import torch
from typer_config.loaders import yaml_loader

from ddw import prepare_data
from ddw.utils.subtomo_dataset import SubtomoDataset
from ddw.utils.visualization import plot_tomo_slices
from ddw.utils.print import pprint_dict, print_help_for_function_arguments

## Running Data Preparation for the BrainWeb Data

As in the original tutorial, there are two ways to run the data preparation step: either by running the `prepare_data` function directly in python, or by running the `ddw prepare-data` command-line command together with a `yaml` config file. We use the latter, with our `brainweb_config.yaml`.

In [ ]:
config = yaml_loader("./brainweb_config.yaml")
project_dir = config["shared"]["project_dir"]
if os.path.exists(project_dir):
    shutil.rmtree(project_dir)

!uv run ddw prepare-data --config ./brainweb_config.yaml

Let's now inspect the sub-tomogram:

In [ ]:
fitting_subtomo = torch.load("./brainweb_project/subtomos/fitting_subtomos/subtomo/0.pt")

plot_tomo_slices(fitting_subtomo).show()

**Note:** Although the subtomogram size in `./brainweb_config.yaml` is set to $64$, the sub-tomograms are actually of size $\approx 91 \approx \sqrt{2} * 64$. As in the original tutorial, this is because for model fitting, the sub-tomograms are randomly rotated. If sub-tomograms of size $64$ were extracted directly, the rotated sub-tomograms would have black borders for some rotations. To avoid this, sub-tomograms are automatically extracted with this larger shape, and then cropped to shape $64$ after rotation during the generation of the model inputs and targets (see next notebook).

In [ ]:
print(fitting_subtomo.shape)

Internally, the command 
```
ddw prepare-data --config ./brainweb_config.yaml
```
calls the function `ddw.prepare_data.prepare_data` with the parameters specified in the `brainweb_config.yaml` file. We first discuss the most important parameters of the `prepare_data` function, and then show how they are specified in the config file.

## Most Important Parameters of `prepare_data` 

You can find a full list of parameters for `ddw prepare-data` by executing
```
ddw prepare-data --help
```
in the commandline. Here, we only discuss the most important parameters.


The most important parameters are the tomogram (here, the average of the two BrainWeb reconstructions with independent projections) from which the sub-tomogram are extracted, and the size of the sub-tomograms:

In [ ]:
print_help_for_function_arguments(prepare_data, print_only_required=True)


Another important (but not required) parameter are 3 stride values for the 3D sliding window procedure used to extract the sub-tomograms. Using smaller strides will result in more sub-tomograms, but also in more overlap between them. For the BrainWeb data, we use smaller strides `[16, 19, 16]` (vs. `[64, 80, 80]` in the original tutorial), since BrainWeb volumes are smaller than the original tilt series reconstruction.

In [ ]:
print_help_for_function_arguments(prepare_data, arg_names=["subtomo_extraction_strides"])

**Note on masks:** the original tutorial also supports restricting sub-tomogram extraction to a region of interest (ROI) using binary `.mrc` masks (`mask_files` / `min_nonzero_mask_fraction_in_subtomo`). We do **not** use a mask for the BrainWeb data — sub-tomograms are extracted from the full volume.

## The `brainweb_config.yaml` File

While all parameters of the `prepare_data` function can be specified directly in python or in the commandline, we recommend using a `yaml` config file for better readability and reproducibility. Using our `./brainweb_config.yaml`, we now explain how parameters are specified for `prepare_data`.

In [ ]:
config = yaml_loader("./brainweb_config.yaml")
print(config.keys())

The important config entries for data preparation are:
- `prepare_data`: Parameters that are specific to the data preparation step.
- `shared`: Parameters that are shared between all steps of the pipeline. Note that some of these parameters are not used in `prepare_data` but for other steps. All parameters specified in `shared` can be overwritten by the specific step parameters.

In [ ]:
pprint_dict(config["prepare_data"])

In [ ]:
pprint_dict(config["shared"])